In [9]:
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
from openai import OpenAI
openai_client = OpenAI()

In [11]:
def llm(prompt):
  response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=prompt
  )
  return response.output_text

In [ ]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

In [13]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

edit on GitHub
#Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

edit on GitHub
#What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the announcements channel on Telegram &amp; Slack before it begins. You can also watch live on the DataTalksClub YouTube Channel.

Don’t post questions in chat as they may be missed if the room is very active.

edit on GitHub
#Cloud alternatives with GPU
Check the quota and reset cycle carefully. Is the free hours limit per month or per week? Usually, if you change the configuration, the free hours quota might also be adjusted, or it might be billed separately.

Potential options include:

Google Colab
Kaggle
Databricks (possibly)
Consider using GPTs to discover more options. Be aware that some platforms might have restrictions on what you can and cannot install, so ensure to read what is included in the free vs paid tier.

edit on GitHub
#Leaderboard: I am not on the leaderboard / how do I know which one I am on the leaderboard?
When you set up your account, you are automatically assigned a random name, such as “Lucid Elbakyan.” Click on the "Jump to your record on the leaderboard" link to find your entry.

If you want to see what your Display name is, click on the "Edit Course Profile" button.

image #1

First field: This is your nickname/displayed name. You can change it if you want to be known by your Slack username, GitHub username, or any other nickname of your choice. This is useful if you want to remain anonymous.
Second field: Change this to your official name as in your identification documents—passport, national ID card, driver's license, etc. This is mandatory if you do not want "Lucid Elbakyan" on your certificate. This name will appear on your Certificate!
edit on GitHub
#Certificate: Can I follow the course in a self-paced mode and get a certificate?
No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.

edit on GitHub
#I missed the first homework - can I still get a certificate?
Yes, you need to pass the Capstone project to get the certificate. Homework is not mandatory, though it is recommended for reinforcing concepts, and the points awarded count towards your rank on the leaderboard.

edit on GitHub
#Homework: Why does the content keep changing?
This course is being offered for the first time, and things will keep changing until a given module is ready, at which point it shall be announced. Working on the material or homework in advance will be at your own risk, as the final version could be different.

edit on GitHub
#When will the course be offered next?
Summer 2025.

edit on GitHub
#Are there any lectures/videos? Where are they?
Please check the bookmarks and pinned links, especially DataTalks.Club’s YouTube account.

edit on GitHub
#WSL2: ResponseError: model requires more system memory (X.X GiB) than is available (Y.Y GiB). My system has more than X.X GiB.
Your WSL2 is set to use Y.Y GiB, not all your computer memory. To allocate more RAM, follow these steps:

Create a .wslconfig file under your Windows user profile directory: "C:/Users/YourUsername/.wslconfig".

Include the desired RAM allocation in the file:

[wsl2]
memory=8GB
Restart WSL using the command:

wsl --shutdown
Run the free command in WSL to verify the changes.

For more details, read this article.

edit on GitHub
#Server Error (500) When logging in to course homework using GitHub
Additional error text seen:

Third-Party Login Failure

An error occurred while attempting to login via your third-party account.
The current solution is to use Google or Slack to log in and submit homework answers, as the root cause analysis for the GitHub issue is sporadic and doesn’t impact all users.

edit on GitHub
'''

In [ ]:
prompt = f'''
"

Question:
{question}

Context:
{context}
'''

In [ ]:
print(prompt)

In [16]:
question = 'I just discovered the course. Can I join now?'
answer = llm(prompt)
print(answer)

Yes, but if you want to receive a certificate, you need to submit your project while they’re still accepting submissions.


In [26]:
def rag(question):
  search_results = search(question)
  user_prompt = build_prompt(question, search_results)
  return llm(user_prompt)

In [18]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [19]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
  course_url = f'{url_prefix}{course['path']}'

  course_response = requests.get(course_url)
  course_response.raise_for_status()
  course_data = course_response.json()

  documents.extend(course_data)

len(documents)

1349

In [7]:
from minsearch import Index

index = Index(
  text_fields=['question','section', 'answer'],
  keyword_fields = ['course']
)

index.fit(documents)

In [ ]:
index.search(
  question, 
  filter_dict={'course': 'llm-zoomcamp'}, 
  num_results=5
  )

In [ ]:
def search(question, course='llm-zoomcamp'):
  return index.search(
    question,
    boost_dict={'question': 2.0},
    filter_dict={'course': course},
    num_results=5
  )

In [25]:
search_results = search(question)

In [27]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers.
If the answer is not found in the context, respond with "I don't know.
'''

In [33]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [28]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
context = build_context(search_results)
print(context)

In [31]:
def build_prompt(question, search_results):
  context = build_context(search_results)
  prompt = USER_PROMPT_TEMPLATE.format(
    question=question, 
    context=context
    )
  return prompt.strip()

In [34]:
prompt = build_prompt(question, search_results)

In [35]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project